In [8]:
import os
import requests
import pandas as pd

from io import BytesIO
from pathlib import Path
from dotenv import load_dotenv
from azure.storage.filedatalake import DataLakeServiceClient


# 1. DEFINIR API E PERÍODO

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/"
    "versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)

parametros = {
    "@dataInicial": "'01-01-2025'",
    "@dataFinalCotacao": "'10-19-2026'",
    "$format": "json"
}


# 2. CONSULTAR API BACEN

resposta = requests.get(
    url,
    params=parametros,
    timeout=60
)

resposta.raise_for_status()

dados = resposta.json()


# 3. CONVERTER DADOS PARA DATAFRAME

df = pd.DataFrame(dados["value"])

print("Registros extraídos:", len(df))

if df.empty:
    raise ValueError("Nenhuma cotação encontrada.")


# 4. DEFINIR CAMINHO DO ARQUIVO LOCAL

arquivo = Path("data/arquivos_api/cotacao_dolar_historico.csv")

arquivo.parent.mkdir(
    parents=True,
    exist_ok=True
)


# 5. SALVAR ARQUIVO CSV

df.to_csv(
    arquivo,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo CSV salvo em:", arquivo.resolve())


# 6. CONVERTER PARA PARQUET

buffer = BytesIO()

df.to_parquet(
    buffer,
    index=False,
    engine="pyarrow"
)


# 7. CARREGAR CREDENCIAL DO AZURE

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

if not connection_string:
    raise ValueError("Credencial do Azure não encontrada no .env")


# 8. CONECTAR AO AZURE DATA LAKE

datalake_service = DataLakeServiceClient.from_connection_string(
    connection_string
)

file_system = datalake_service.get_file_system_client(
    file_system="bronze"
)


# 9. ENVIAR HISTÓRICO PARA BRONZE

arquivo_azure = file_system.get_file_client(
    "api/bacen/cotacao_dolar_historico.parquet"
)

arquivo_azure.upload_data(
    buffer.getvalue(),
    overwrite=True
)

print("Histórico enviado para Bronze com sucesso!")
print("Registros enviados:", len(df))


Registros extraídos: 452
Arquivo CSV salvo em: C:\Users\vitos\OneDrive\Documentos\BI\bi_ecommerce\data\arquivos_api\cotacao_dolar_historico.csv
Histórico enviado para Bronze com sucesso!
Registros enviados: 452
